# PC-FNO3D NATIONAL v4, candidat D (C' + entrée ANEO à l'entraînement) — entraînement, graine 44 (Colab, A100)

Règle `configs/national/regle_v4_national.json` (sha256 `60671e843ee6…`), vérifiée par le carnet. **Uniquement avec l'accord écrit d'Ines.** Vérité d'entraînement v3 (`cache_v3_partie_*`).

Aucune lecture du test dans ce carnet.

## 1. Drive, code et données

In [ ]:
from google.colab import drive
import os as _os
if not _os.path.isdir("/content/drive/MyDrive"):   # deja monte (panneau Fichiers) : ne pas redemander
    drive.mount("/content/drive")

import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

GT = Path("/content/drive/MyDrive/pcfno3d_grand_tunis")   # code et poids du Grand Tunis
NAT = Path("/content/drive/MyDrive/pcfno3d_national")
NAT.mkdir(parents=True, exist_ok=True)
JOURNAUX = NAT / "journaux"
JOURNAUX.mkdir(exist_ok=True)
libre = shutil.disk_usage("/content/drive/MyDrive").free / 1e9
print(f"place libre sur le Drive : {libre:.0f} Go (le jeu national en demande ~50)")

CODE = Path("/content/PFE")
if CODE.exists():
    shutil.rmtree(CODE)
CODE.mkdir(parents=True)
with zipfile.ZipFile(GT / "pcfno3d_code.zip") as zf:
    zf.extractall(CODE)
tampon = CODE / "PCFNO3D_VERSION.txt"
print(tampon.read_text().strip() if tampon.exists() else "tampon : absent")
%cd /content/PFE
!pip install -q --no-deps -e . 2>&1 | tail -2
!pip install -q pandera xarray netCDF4 h5netcdf 2>&1 | tail -2
sys.path[:0] = ["/content/PFE", "/content/PFE/src"]

def lancer(nom: str, commande: str) -> None:
    """Commande obligatoire ; sortie affichee ET journalisee sur le Drive ; arret si echec."""
    t0 = time.time()
    with open(JOURNAUX / f"{nom}.log", "a", encoding="utf-8") as log:
        log.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} : {commande}\n")
        p = subprocess.Popen(commande, shell=True, cwd="/content/PFE", stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for ligne in p.stdout:
            print(ligne, end="")
            log.write(ligne)
            log.flush()
        p.wait()
    print(f"[{nom}] {time.time() - t0:.0f} s, code {p.returncode}")
    if p.returncode:
        raise RuntimeError(f"{nom} a echoue (code {p.returncode}) : voir {JOURNAUX / (nom + '.log')}")

# donnees : grille nationale, grille_reelle.json, forcage ERA5 (une archive sur le Drive)
DON = Path("/content/donnees")
G = DON / "national"
if not (G / "forcage_era5.csv").exists():
    with zipfile.ZipFile(NAT / "national_donnees.zip") as zf:
        zf.extractall(DON)
print(sorted(p.name for p in G.iterdir()))

# code du depot public au commit fige (remplace l'archive du Drive, plus ancienne)
COMMIT = "77a058b0649f77648837559ba3f9b807cbe5a41f"
os.chdir("/content")    # le noyau etait dans /content/PFE : le supprimer laisserait pip dans un dossier efface
subprocess.run(f"rm -rf /content/PFE && git init -q /content/PFE && cd /content/PFE && git remote add origin "
               f"https://github.com/ines-rouatbi/pcfno3d.git && git fetch -q --depth 1 origin {COMMIT} "
               f"&& git checkout -q FETCH_HEAD", shell=True, check=True)
os.chdir("/content/PFE")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "/content/PFE"],
                   capture_output=True, text=True)
if r.returncode:
    print(r.stdout[-3000:], r.stderr[-3000:])
    raise SystemExit("pip install -e /content/PFE a echoue (sortie ci-dessus)")
import importlib
importlib.invalidate_caches()
import hashlib
for f, h in (("regle_v4_national.json", "60671e843ee648ee2422168a7659a53dfe8aa414be7b1c7c6f54eeec622a43f8"),):
    assert hashlib.sha256((CODE / "configs" / "national" / f).read_bytes()).hexdigest() == h, f
print("code", COMMIT, "| regle v4 verifiee")

## 2. Cache v3 et contrôle de la vérité

In [ ]:
LOC = Path("/content/national")
CACHE, JEU = LOC / "cache", LOC / "jeu"
for k in (1, 2, 3):
    if not (NAT / f"cache_v3_partie_{k}.zip").exists():
        raise FileNotFoundError(f"cache_v3_partie_{k}.zip absent : finir le carnet de generation")
if not (CACHE / "index.csv").exists():
    CACHE.mkdir(parents=True, exist_ok=True)
    for k in (1, 2, 3):
        with zipfile.ZipFile(NAT / f"cache_v3_partie_{k}.zip") as zf:
            zf.extractall(CACHE)
    with zipfile.ZipFile(NAT / "jeu_v3.zip") as zf:
        zf.extractall(JEU)
    import pandas as pd
    idx = pd.concat([pd.read_csv(CACHE / f"index_partie_{k}.csv") for k in (1, 2, 3)])
    idx.drop_duplicates("stem").to_csv(CACHE / "index.csv", index=False)
import pandas as pd
print(pd.read_csv(CACHE / "index.csv")["split"].value_counts().to_string())
lancer("controle", "python -u -m pcfno3d -c national_1km_colab check")

## 3. Mémoire du GPU

In [ ]:
# Essai de memoire dans un PROCESSUS SEPARE : le noyau du carnet ne touche jamais au GPU (le 11/11, l'essai fait
# dans le noyau y gardait 3,3 Go et l'entrainement sur L4 22 Go a manque de memoire). Allocateur a segments
# extensibles (moins de fragmentation), herite par les commandes lancees ensuite. Aucun effet sur les calculs.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
ESSAI = r"""
import json, sys, torch
from pcfno3d.config import Config
from pcfno3d.geometry import Geometry
from pcfno3d.model import build_model
recalcul = sys.argv[1] == "1"
cfg = Config.load("configs/national_1km_colab.yaml")
cfg.model.recalcul_activations = recalcul
geom = Geometry(cfg, device="cuda")
m = build_model(cfg, geom.c_in).cuda().train()
x = torch.randn(cfg.train.batch_size, geom.c_in, *geom.shape, device="cuda")
torch.cuda.reset_peak_memory_stats()
try:
    with torch.autocast("cuda", dtype=torch.bfloat16):
        m(x).float().pow(2).mean().backward()
    pic = torch.cuda.max_memory_allocated() / 1e9
except torch.cuda.OutOfMemoryError:
    pic = None
p = torch.cuda.get_device_properties(0)
print("ESSAI", json.dumps({"gpu": p.name, "total": p.total_memory / 1e9, "pic": pic}))
"""
Path("/content/essai_memoire.py").write_text(ESSAI)
def essai(recalcul: bool) -> dict:
    r = subprocess.run([sys.executable, "/content/essai_memoire.py", "1" if recalcul else "0"], cwd="/content/PFE",
                       capture_output=True, text=True)
    ligne = next((l for l in r.stdout.splitlines() if l.startswith("ESSAI ")), None)
    if ligne is None:
        print(r.stdout[-2000:], r.stderr[-2000:])
        return {"gpu": "?", "total": float("nan"), "pic": None}
    return json.loads(ligne[6:])
e = essai(False)
total = e["total"]
RECALCUL = e["pic"] is None or e["pic"] > 0.85 * total
if RECALCUL:
    e = essai(True)
print(e["gpu"], f"{total:.0f} Go | pic de l'essai {e['pic'] if e['pic'] is None else round(e['pic'], 1)} Go "
      f"| recalcul des activations : {RECALCUL}")
if total < 30:
    print("ATTENTION : GPU de moins de 30 Go (L4 ?). Le national a manque de memoire sur L4 le 11/11 avec l'ancien "
          "carnet ; cet essai ne compte ni l'optimiseur ni l'EMA. Preferer une A100.")
OPTS = f"--set model.recalcul_activations={str(RECALCUL).lower()}"

## 4. Entraînement (validation seulement), fin de la session

In [ ]:
import shlex
def choisir_poids(src):
    """Le Drive garde parfois deux fichiers du meme nom (grand_tunis_v4_dso : deux best.pt de 1,69 Go,
    vus le 31/10) ; Colab peut n'en montrer qu'un, au hasard. On retient celui dont l'epoque est celle
    du test evalue (eval_pcfno3d/best_wcont, sinon best), meme s'il n'y en a qu'un : jamais un poids
    pris au hasard."""
    cands = sorted(src.glob("best*.pt"))
    if not cands:
        return src / "best.pt"
    attendu = next((json.loads(f.read_text()).get("epoque") for d in ("best_wcont", "best")
                    if (f := src / "eval_pcfno3d" / d / "test_summary.json").exists()), None)
    import torch
    epoques = {}
    for c in cands:
        ck = torch.load(str(c), map_location="cpu", mmap=True, weights_only=False)
        epoques[c] = ck.get("epoch", -2) + 1
        del ck
    print("poids candidats :", {c.name: e for c, e in epoques.items()}, "| epoque evaluee :", attendu)
    ok = [c for c in cands if epoques[c] == attendu] if attendu is not None else cands
    if not ok:
        raise RuntimeError(f"aucun poids de {src} n'a l'epoque evaluee ({attendu}) : verifie sur le Drive")
    return max(ok, key=lambda c: c.stat().st_mtime)          # a epoque egale : le plus recent

import shlex
from google.colab import runtime
import traceback
GRAINE = 44
CANDIDAT = "D"                     # Cp ou D (regle_v4_national.json)
RUN = NAT / "runs" / f"national_1km_v4_{CANDIDAT}_s{GRAINE}"
# entrainement interrompu (session Colab coupee, last.pt local perdu : plan A) : le dossier est mis de cote,
# renomme et JAMAIS supprime ni ecrase, puis l'entrainement repart de zero sous le nom de la regle
if (RUN / "history.json").exists() and not (RUN / "info_capteurs_validation.json").exists():
    cote = RUN.with_name(RUN.name + "_interrompu_" + time.strftime("%Y%m%d_%H%M%S"))
    shutil.move(str(RUN), str(cote))
    print("entrainement interrompu mis de cote :", cote)
RUN.mkdir(parents=True, exist_ok=True)
EXTRA = "--set loss.lambda_rough=1.0"
# parametres de la degradation d'ERA5 (jours d'entrainement seulement), ecrits AVANT tout entrainement ;
# communs a D (entrainement) et a l'evaluation d'information de C' et D (validation)
DEGV4 = NAT / "runs" / "degradation_era5_v4.json"
if not DEGV4.exists():
    lancer("degradation_era5_v4", f"python -u scripts/degradation_era5.py --donnees {G} "
                                  f"--calendrier {NAT / 'calendrier.json'} --sortie {DEGV4}")
if CANDIDAT == "D":
    DEG = RUN / "degradation_era5.json"
    if not DEG.exists():
        shutil.copy2(DEGV4, DEG)
    EXTRA += (f" --set mask.aneo_degradation={shlex.quote(str(DEG))} --set mask.aneo_p_zone=0.5"
              f" --set {shlex.quote('mask.aneo_capteurs=[20,200]')} --set mask.aneo_hauteur_capteur_m=25")
INIT = shlex.quote(str(choisir_poids(NAT / "runs" / "national_1km_v2")))
print("depart :", INIT, "| run :", RUN)
try:
    SEL = shlex.quote('train.select_criteres=["dso1_rmse","dso2_masse","dso3_rugosite"]')
    lancer(f"entrainer_v4_{CANDIDAT}_s{GRAINE}",
           f"python -u -m pcfno3d -c national_1km_colab {OPTS} {EXTRA} --set {SEL} --set train.sync_last=false "
           f"--set train.init_from={INIT} --set train.lr=1e-4 --set train.seed={GRAINE} "
           f"--set paths.run_dir={shlex.quote(str(RUN))} --set paths.local_dir=/content/run_local_v4_{CANDIDAT}_s{GRAINE} train")
    # AUCUNE lecture du test ici (regle_v4_national.json) : seul le carnet de test lit le test, une fois
    print((RUN / "history.json").read_text()[-3000:])
    # information, sans effet sur le choix : validation propre / ERA5 degradee / + 50 capteurs simules
    if not (RUN / "info_capteurs_validation.json").exists():
        lancer(f"info_capteurs_v4_{CANDIDAT}_s{GRAINE}",
               f"python -u scripts/info_capteurs_v4.py -c national_1km_colab {OPTS} --run {shlex.quote(str(RUN))} "
               f"--degradation {shlex.quote(str(DEGV4))}")
except Exception:
    traceback.print_exc()
finally:
    runtime.unassign()